# A_15 — "Fatores de risco" no SIVEP: as comorbidades em branco são "não"?

**Hipótese.** A ficha tem a pergunta-mãe `fator_risc` ("possui fatores de risco/comorbidades?"). Se a lista de comorbidades só é preenchida quando a resposta é "sim", o branco numa comorbidade com `fator_risc` = não significa "não tem", e não "não informado".

**Confirma a hipótese** se, com `fator_risc` = não, o branco for quase 100% e, com `fator_risc` = sim, bem menor (tabela 1).

Este notebook só lê `coorte_final.parquet` (colunas necessárias) e **não treina nada**. Comente as três tabelas no card com a proposta de regra.

In [ ]:
from pathlib import Path
import pandas as pd

from src.utils import config as C

COORTE = C.ANALYTICAL_DIR / "coorte_final.parquet"
SAIDA = C.ROOT_DIR / "outputs" / "tables"
SAIDA.mkdir(parents=True, exist_ok=True)

# As 12 comorbidades + puerpera
COMORBIDADES = [
    "cardiopati", "hematologi", "sind_down", "hepatica", "asma",
    "diabetes", "neurologic", "pneumopati", "imunodedepre", "renal",
    "obesidade", "out_morbi", "puerpera",
]

COLUNAS = ["fator_risc", "ano_internacao", "alvo_obito_srag"] + COMORBIDADES
df = pd.read_parquet(COORTE, columns=COLUNAS)

print(f"[A15] {len(df):,} fichas carregadas")
print(f"[A15] Anos: {sorted(df['ano_internacao'].dropna().unique().tolist())}")

In [ ]:
# ============================================================
# Célula 2 — Entender os valores de fator_risc
# ============================================================
# Os valores podem variar por ano: 1/2/9, S/N ou texto.
# Anote o mapeamento antes de classificar.
print(df.groupby("ano_internacao")["fator_risc"].value_counts(dropna=False).to_string())

In [ ]:
# ============================================================
# Célula 3 — Classificar fator_risc (após conferir a célula 2)
# ============================================================
def classificar_fator_risc(s: pd.Series) -> pd.Series:
    """Converte fator_risc em sim/nao/ignorado/branco.

    Confirmar o mapeamento com a saída da célula 2 antes de rodar.
    """
    v = s.astype("string").str.strip().str.upper()
    sim = v.isin(["1", "1.0", "S", "SIM"])
    nao = v.isin(["2", "2.0", "N", "NAO", "NÃO"])
    ign = v.isin(["9", "9.0", "IGNORADO"])
    return pd.Series(
        ["sim" if s_ else "nao" if n_ else "ignorado" if i_ else "branco"
         for s_, n_, i_ in zip(sim, nao, ign)],
        index=s.index,
    )

df["fator_risc_cat"] = classificar_fator_risc(df["fator_risc"])
print(df["fator_risc_cat"].value_counts(dropna=False))

In [ ]:
# ============================================================
# Célula 4 — Tabela 1: fator_risc × comorbidade em branco
# Hipótese confirmada se, com fator_risc=não, o branco for ~100%,
# e com fator_risc=sim, for bem menor.
# ============================================================
def em_branco(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Branco = NaN ou string vazia. Código 9 (ignorado) NÃO conta como branco."""
    return df[cols].isna() | (df[cols].astype("string").apply(lambda c: c.str.strip()) == "")

branco = em_branco(df, COMORBIDADES)
branco["ano"] = df["ano_internacao"].values
branco["fator"] = df["fator_risc_cat"].values

long = branco.melt(id_vars=["ano", "fator"], var_name="comorbidade", value_name="branco")
tabela1 = (
    long.groupby(["comorbidade", "ano", "fator"])["branco"]
    .agg(n="size", pct_branco=lambda s: s.mean() * 100)
    .round(2)
    .reset_index()
)
tabela1.to_csv(SAIDA / "eda_02_fator_risc_comorbidades.csv", index=False, encoding="utf-8-sig")

# Resumo de leitura (2023–2025)
resumo = (
    tabela1[tabela1["ano"].isin([2023, 2024, 2025])]
    .pivot_table(index="comorbidade", columns="fator", values="pct_branco", aggfunc="mean")
    .round(1)
)
print(resumo.to_string())

In [ ]:
# ============================================================
# Célula 5 — Tabela 2: mortalidade por fator_risc, por ano
# ============================================================
tabela2 = (
    df.groupby(["ano_internacao", "fator_risc_cat"])["alvo_obito_srag"]
    .agg(n="size", obitos="sum")
    .reset_index()
)
tabela2["mortalidade_pct"] = (tabela2["obitos"] / tabela2["n"] * 100).round(2)
tabela2.to_csv(SAIDA / "eda_02_fator_risc_mortalidade.csv", index=False, encoding="utf-8-sig")
print(tabela2[tabela2["ano_internacao"].isin([2023, 2024, 2025])].to_string(index=False))

In [ ]:
# ============================================================
# Célula 6 — Tabela 3: ausência por desfecho, só entre fator_risc=sim
# Se a diferença de 13–21 p.p. cair muito, parte do "viés de
# documentação" era a regra da ficha.
# ============================================================
ANOS_REF = [2023, 2024, 2025]
saidas = []

for rotulo, sub in [("todas", df), ("so_fator_sim", df[df["fator_risc_cat"] == "sim"])]:
    sub = sub[sub["ano_internacao"].isin(ANOS_REF)]
    b = em_branco(sub, COMORBIDADES)
    for c in COMORBIDADES:
        cura = b.loc[sub["alvo_obito_srag"] == 0, c].mean() * 100
        obito = b.loc[sub["alvo_obito_srag"] == 1, c].mean() * 100
        saidas.append({
            "populacao": rotulo,
            "comorbidade": c,
            "n": len(sub),
            "pct_branco_cura": round(cura, 2),
            "pct_branco_obito": round(obito, 2),
            "dif_pp_cura_menos_obito": round(cura - obito, 2),
        })

tabela3 = pd.DataFrame(saidas)
tabela3.to_csv(SAIDA / "eda_02_fator_risc_ausencia_desfecho.csv", index=False, encoding="utf-8-sig")
print(tabela3.pivot_table(index="comorbidade", columns="populacao", values="dif_pp_cura_menos_obito").round(1).to_string())

In [ ]:
# ============================================================
# Célula 7 — Proposta de regra (salvar em texto)
# ============================================================
proposta = """
Proposta de regra (se a hipótese se confirmar):

1. Comorbidade em branco e fator_risc = não  → "2" (não).
2. Comorbidade em branco e fator_risc = sim,
   ignorado ou branco                        → continua AUSENTE.

fator_risc NÃO entra como variável do modelo por padrão
(redundante com as comorbidades recodificadas).
Decisão final no #140 com a ablação da Pessoa C.
"""
(SAIDA / "eda_02_fator_risc_proposta.txt").write_text(proposta, encoding="utf-8")
print(proposta)